# PIIMiddleware

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

chat_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", )

In [3]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=chat_model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
帮我向 156168188@qq.com 发送一封邮件
同时查看银行卡号： 5105-1051-0510-5100 的余额
访问 https://localhost:12345
确认这是不是 MAC地址： 11-11-11-11-11-11
""")]
})
for msg in response["messages"]:
    msg.pretty_print()
try:
    response1 = agent.invoke({
        "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

================================ Human Message =================================


帮我向 [REDACTED_EMAIL] 发送一封邮件
同时查看银行卡号： ****-****-****-5100 的余额
访问 <url_hash:dd5fc2a9>
确认这是不是 MAC地址： **-**-**-**-**-11

================================== Ai Message ==================================

我无法替你直接发送邮件、访问银行卡账户查询余额，或打开该网址。为安全起见，也请不要在聊天中提供完整银行卡号、密码、验证码或登录信息。

我可以帮你：

- **起草邮件**：请提供收件人、主题和邮件内容，我会生成可直接复制发送的版本。
- **查询余额**：请通过银行官方 App、官网或客服电话查询；不要点击不明链接或向他人透露验证码。
- **判断 MAC 地址格式**：`**-**-**-**-**-11` 符合常见 MAC 地址的外观格式——6 组、每组 2 个十六进制字符（0–9、A–F）。但由于前五组被隐藏，无法确认它是否对应一个真实有效的 MAC 地址。
============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content
